# Price Statistics – Summary Tables
stETH discount/premium summary statistics.

In [1]:
import stata_setup
stata_setup.config('/home/yichen', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               University College London

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off

global PROCESSED_DATA "../../processed_data"
global TABLES         "../../lido-bank/tables"


. 
. clear all

. set more off

. set varabbrev off

. 
. global PROCESSED_DATA "../../processed_data"

. global TABLES         "../../lido-bank/tables"

. 


In [3]:
%%stata
****************************
* Table: stETH Discount / Premium Summary Statistics
* Cols: Full Sample | Lido V1 (no withdrawal) | Lido V2 (withdrawal enabled)
****************************

import delimited using "$PROCESSED_DATA/steth_eth_daily.csv", varnames(1) clear

gen date2 = date(date, "YMD")
format date2 %td

* Lido V2 upgrade: May 15, 2023
local v2_date = date("2023-05-15", "YMD")
gen is_v2 = (date2 >= `v2_date')

replace discount_mean = discount_mean * 100

* ── Helper macro: compute stats into locals for a given subsample ─────────────
* Usage: call for each of: all, v1 (is_v2==0), v2 (is_v2==1)

foreach period in all v1 v2 {

    if "`period'" == "all" local cond ""
    if "`period'" == "v1"  local cond "& is_v2 == 0"
    if "`period'" == "v2"  local cond "& is_v2 == 1"

    quietly count if 1 == 1 `cond'
    local N_`period' = r(N)

    quietly summarize steth_volume if 1 == 1 `cond'
    local avg_vol_`period' : display %9.3f r(mean) / 1e9

    quietly summarize is_discount if 1 == 1 `cond'
    local pct_disc_`period' : display %6.1f r(mean) * 100

    quietly summarize is_premium if 1 == 1 `cond'
    local pct_prem_`period' : display %6.1f r(mean) * 100

    quietly summarize discount_mean if is_discount == 1 `cond', detail
    local avg_disc_`period'  : display %7.4f r(mean)
    local med_disc_`period'  : display %7.4f r(p50)

    quietly summarize discount_mean if is_premium == 1 `cond', detail
    local avg_prem_`period'  : display %7.4f r(mean)
    local med_prem_`period'  : display %7.4f r(p50)
}

* ── Write LaTeX ──────────────────────────────────────────────────────────────
tempname fh
file open `fh' using "$TABLES/sum_steth_discount.tex", write replace

file write `fh' "{\small" _n
file write `fh' "\begin{tabular}{lccc}" _n
file write `fh' "\toprule" _n
file write `fh' " & Full Sample & Lido V1 & Lido V2 \\" _n
file write `fh' " & (Dec 2020--Apr 2026) & (Dec 2020--Apr 2023) & (May 2023--Apr 2026) \\" _n
file write `fh' "\midrule" _n
file write `fh' "Observations (days) & `N_all' & `N_v1' & `N_v2' \\" _n
file write `fh' "\midrule" _n
file write `fh' "Avg Daily Volume (USD bn) & `avg_vol_all' & `avg_vol_v1' & `avg_vol_v2' \\" _n
file write `fh' "Proportion of Discount Days (\%) & `pct_disc_all' & `pct_disc_v1' & `pct_disc_v2' \\" _n
file write `fh' "Proportion of Premium Days (\%)  & `pct_prem_all' & `pct_prem_v1' & `pct_prem_v2' \\" _n
file write `fh' "\midrule" _n
file write `fh' "Average Discount (\%)  & `avg_disc_all' & `avg_disc_v1' & `avg_disc_v2' \\" _n
file write `fh' "Median Discount (\%)   & `med_disc_all' & `med_disc_v1' & `med_disc_v2' \\" _n
file write `fh' "Average Premium (\%)   & `avg_prem_all' & `avg_prem_v1' & `avg_prem_v2' \\" _n
file write `fh' "Median Premium (\%)    & `med_prem_all' & `med_prem_v1' & `med_prem_v2' \\" _n
file write `fh' "\bottomrule" _n
file write `fh' "\end{tabular}}" _n
file close `fh'

display "Done. N_all=`N_all'  N_v1=`N_v1'  N_v2=`N_v2'"


. ****************************
. * Table: stETH Discount / Premium Summary Statistics
. * Cols: Full Sample | Lido V1 (no withdrawal) | Lido V2 (withdrawal enabled)
. ****************************
. 
. import delimited using "$PROCESSED_DATA/steth_eth_daily.csv", varnames(1) cle
> ar
(encoding automatically selected: ISO-8859-1)
(16 vars, 1,931 obs)

. 
. gen date2 = date(date, "YMD")

. format date2 %td

. 
. * Lido V2 upgrade: May 15, 2023
. local v2_date = date("2023-05-15", "YMD")

. gen is_v2 = (date2 >= `v2_date')

. 
. replace discount_mean = discount_mean * 100
(1,931 real changes made)

. 
. * ── Helper macro: compute stats into locals for a given subsample ──────────
> ───
. * Usage: call for each of: all, v1 (is_v2==0), v2 (is_v2==1)
. 
. foreach period in all v1 v2 {
  2. 
.     if "`period'" == "all" local cond ""
  3.     if "`period'" == "v1"  local cond "& is_v2 == 0"
  4.     if "`period'" == "v2"  local cond "& is_v2 == 1"
  5. 
.     quietly count if 1 == 1 `cond'
  